# Normalization for T1 $\to$ T1ce: matching the *unenhanced* brain

**Where this started.** T1 and T1ce carry the same anatomy at roughly the same intensity
distribution, except that the enhancing tumor adds a high-intensity tail to T1ce. Any statistic
estimated over the whole brain absorbs part of that tail, so it is set partly by the tumor and not
only by the anatomy the two contrasts share -- and they end up on slightly different scales for a
reason that has nothing to do with the tissue being matched.

**What worked.** Of the schemes below, `median-mad` -- centre on the median, divide by
$1.4826\times\mathrm{MAD}$ -- came out best. Both estimators are robust, so the tail moves them far
less than it moves the mean and the std.

**The question now.** `median-mad` is robust to the tail but still *estimated over it*. If the
premise is that only the **unenhanced** brain should match between T1 and T1ce, the statistics
should be estimated over $\mathrm{brain}\setminus\mathrm{ET}$ and nothing else. That is what the
`-noet` schemes do: the normalization is still applied to every voxel, only the *estimation region*
changes.

| scheme | centre | scale | estimated over | why it is here |
|---|---|---|---|---|
| `zscore` | mean | std | brain | what `preprocessing/cmap.py` shipped originally |
| `zscore-noet` | mean | std | brain $\setminus$ ET | **the mean, given the mask** -- the estimator with the most to gain |
| `zscore-noetce` | mean | std | brain $\setminus$ ET for T1ce, brain elsewhere | the deployable half of it |
| `median-std` | median | std | brain | mean $\to$ median only |
| `median-mad` | median | MAD | brain | **the current best**; the thing to beat |
| `median-mad-noet` | median | MAD | brain $\setminus$ ET | match on unenhanced brain, every contrast |
| `median-mad-noetce` | median | MAD | brain $\setminus$ ET for T1ce, brain elsewhere | only T1ce has an enhancement tail -- is correcting it alone enough? |

(`median-std-noet` and `median-iqr` / `median-iqr-noet` are implemented too; add them to `SCHEMES`.)

`-noetce` is worth separating from `-noet` in both families because they answer different questions.
Excluding ET from *every* contrast keeps all four estimated over the same voxel population, which is
what "the same anatomy" means. Excluding it from T1ce alone targets the tail where it exists, but
then T1 and T1ce are no longer estimated over the same voxels, which reintroduces a mismatch of its
own -- and it is the only one of the two that survives to inference, since T1/FLAIR/T2 never needed
the mask.

### The mean is where the mask should earn its keep

MAD has a **breakdown point of 50%** -- it is unaffected by arbitrary corruption of up to half the
sample -- and the median is the same. ET is a few percent of the brain, so `median-mad-noet` may
come out **numerically indistinguishable** from `median-mad`, in which case the proposal is a no-op
and the correct conclusion is "MAD already did this".

The **mean and the std have no breakdown point at all**. A tail of $f$ voxels sitting $\delta$ above
the bulk shifts the mean by $f\delta$ and inflates the variance by roughly $f\delta^2$, however small
$f$ is. So `zscore` $\to$ `zscore-noet` is where an ET mask can actually change the answer, and the
open question is whether it changes it *enough* -- whether the mask rescues the mean to where the
median already was without one. That comparison is the point of the two families sitting in the same
table, and the section *Does excluding ET move the estimator at all?* quantifies it directly.

### Read the residual tables with care

The `-noet` schemes are *constructed* to align T1 and T1ce over $\mathrm{brain}\setminus\mathrm{ET}$
-- the same region the `bias` and `bg_rms` columns are measured over. They therefore start with an
advantage that is partly definitional, and `bias` $\to 0$ is not on its own evidence that the scheme
is better. (It is not a strict tautology: matching two marginals sets
$\mathrm{med}(\mathrm{T1ce})-\mathrm{med}(\mathrm{T1})=0$, whereas `bias` is
$\mathrm{med}(\mathrm{T1ce}-\mathrm{T1})$, which is a different quantity. But it is close enough
that ranking on it would be circular.)

Exactly two columns are clean:

* **`et_med`** -- the median of $\Delta$ over ET voxels, in units of the normalized-T1 spread. No
  `-noet` scheme ever looks at an ET voxel, and the reference it is divided by is set over the whole
  brain by every scheme alike, so both halves of it are held out.
* **`sigma`** -- $\varsigma = \mathrm{RMS}(\mathrm{T1ce}-\mathrm{T1})$ over the brain, in the
  scheme's own units. Not a fit target, and it is the quantity that sizes the I2SB schedule
  (see `inspect_beta_max.ipynb`), so it is what the downstream bridge actually consumes.

**`separation` and `leak` are NOT clean**, despite being about the tumor. `separation` is
`et_med / bg_rms`, and `leak` counts healthy voxels against a threshold set by `et_med` -- both put a
fit target in the denominator. A `-noet` scheme that drives `bg_rms` toward zero drives `separation`
toward infinity for that reason alone, with no improvement in tumor contrast whatsoever. Read them,
but rank on `et_med` and `sigma`.

### The percentile family (the earlier hypothesis)

`p99`, `proposed` (the $99-100 f_{ET}$ rule) and `oracle` (99th percentile of brain minus ET) are
still implemented. Add them back to `SCHEMES` to reproduce the earlier comparison; the figures that
are specific to them are skipped automatically when they are absent.

## PER_SLICE

`PER_SLICE = True` gives every slice its own statistics. `False` pools the volume, which is what
`preprocessing/cmap.py` does today (`normalize_per_slice: false`), and is the default here for that
reason. Slice-local scaling breaks comparability along $z$ -- the same tissue can take different
values on adjacent slices -- and it makes the `-noet` schemes much noisier, because on a slice that
clips the edge of the tumor a handful of ET voxels defines the excluded region. Per-slice will also
always look flatter on the residual metrics (more free parameters), so judge the *switch* on
downstream training, not on `bg_rms` here.

In [ ]:
import os, glob
import numpy as np
import h5py
import matplotlib.pyplot as plt
%matplotlib inline

# os.chdir('/scratch/ee2178/ImMAP')   # <-- EDIT to your repo root if needed

ROOT = "/home/ee2178/scratch/ee2178/datasets/BraTS/BraTS2021_DataSet_train"   # <-- EDIT
CONTRASTS = ["flair", "t1", "t1ce", "t2"]      # channel order in the h5 (cmap.yaml: contrasts)
FLAIR, T1, T1CE, T2 = 0, 1, 2, 3

# False = pool the whole volume, which is what preprocessing/cmap.py does today
# (cmap.yaml normalize_per_slice: false). Flip to True to A/B the switch itself.
PER_SLICE = False

# Two families, each with and without the ET mask, so the question "does the mask help?" is asked
# of a NON-robust estimator (mean/std) and a robust one (median/MAD) side by side. The mean is the
# one with something to gain: it has no breakdown point, so the tail moves it and the mask can
# undo that. Also available: "median-std-noet", "median-iqr", "median-iqr-noet". Append "p99",
# "proposed", "oracle" for the older percentile hypothesis; their figures self-skip when absent.
SCHEMES = ["zscore", "zscore-noet", "zscore-noetce",
           "median-std", "median-mad", "median-mad-noet", "median-mad-noetce"]

P_BASE       = 99.0   # base percentile for the "healthy tissue" top (percentile family only)
P_FLOOR      = 50.0   # never let the ET correction drag the percentile below this
MIN_VOX      = 200    # slices with fewer brain voxels than this get no reliable percentile
MIN_STAT_VOX = 200    # fewer voxels than this in an estimation region -> fall back to full brain
DIV_SCALE    = 3.0    # the divisor i2sb_dataset applies (cfg scales); used only to report sigma
SEED         = 0

# Percentile-family members, so the cells specific to them can guard on their presence.
# each ET-masked scheme and the brain-wide twin it is meant to improve on
PAIRS = [("zscore-noet", "zscore"),
         ("zscore-noetce", "zscore"),
         ("median-mad-noet", "median-mad"),
         ("median-mad-noetce", "median-mad"),
         ("median-std-noet", "median-std"),
         ("proposed", "p99")]

PCT = [s for s in ("p99", "proposed", "oracle") if s in SCHEMES]
HAVE_PCT = len(PCT) == 3

subjects = sorted(p for d in sorted(glob.glob(os.path.join(ROOT, "*")))
                  if os.path.isdir(d) for p in glob.glob(os.path.join(d, "*_img.h5")))
if not subjects:
    raise RuntimeError(f"no *_img.h5 under {ROOT}")


def has_keys(p, keys=("img_raw", "mask", "et")):
    with h5py.File(p, "r") as h:
        return all(k in h for k in keys)


usable = [p for p in subjects if has_keys(p)]
print(f"{len(subjects)} subject(s); {len(usable)} with img_raw + mask + et")
print(f"mode: {'PER SLICE' if PER_SLICE else 'per volume'}")
print(f"schemes: {', '.join(SCHEMES)}")
if not HAVE_PCT:
    print("percentile family (p99/proposed/oracle) not selected -- its figures will be skipped")
if not usable:
    raise RuntimeError(
        "need 'img_raw' (cmap.yaml save_raw_image: true) and 'et' (save_seg: true). The stored "
        "'img' is already normalized, so a new normalization cannot be tested on it. Backfill ET "
        "with: python preprocessing/cmap.py --config config/BraTS/cmap.yaml --add-seg-only")
if len(usable) < len(subjects):
    print(f"[warn] {len(subjects) - len(usable)} subject(s) lack img_raw and/or et -- excluded")

## Load one subject

`img_raw` is unnormalized, unclipped, background already zeroed -- so every statistic is taken over
the brain mask, never the whole slice (which is mostly zeros).

In [ ]:
def load_subject(path):
    """-> raw (n,H,W,4) float32 unnormalized, brain (n,H,W) bool, et (n,H,W) bool."""
    with h5py.File(path, "r") as h:
        raw = np.asarray(h["img_raw"]).astype(np.float32)
        brain = np.asarray(h["mask"])[..., 0] > 0.5
        et = np.asarray(h["et"])
        et = (et[..., 0] if et.ndim == 4 else et) > 0.5
    return raw, brain, et


rng = np.random.default_rng(SEED)
SUBJECT = usable[int(rng.integers(len(usable)))]      # <-- or paste a path here to pin one
raw, brain, et = load_subject(SUBJECT)

n = raw.shape[0]
brain_z = brain.reshape(n, -1).sum(1)
et_z = et.reshape(n, -1).sum(1)
f_et_z = et_z / np.maximum(brain_z, 1)
has_et = et_z > 0

print(f"subject : {os.path.basename(os.path.dirname(SUBJECT))}")
print(f"volume  : {n} slices of {raw.shape[1]}x{raw.shape[2]}")
print(f"brain   : {brain.sum():,} voxels    ET: {et.sum():,} "
      f"({et.sum() / max(brain.sum(), 1):.3%} of brain, volume-level)")
print(f"\nper-slice ET fraction over the {int(has_et.sum())}/{n} slices that HAVE ET:")
if has_et.any():
    q = np.percentile(f_et_z[has_et], [0, 25, 50, 75, 100])
    print(f"  min {q[0]:.3%} | p25 {q[1]:.3%} | median {q[2]:.3%} | p75 {q[3]:.3%} | max {q[4]:.3%}")
    print(f"  -> proposed T1ce percentile ranges {P_BASE - 100*q[4]:.2f} .. {P_BASE - 100*q[0]:.2f}")
print(f"{int((~has_et).sum())} slice(s) have NO ET; there all three percentile schemes coincide.")
thin = (brain_z < MIN_VOX) & (brain_z > 0)
if thin.any():
    print(f"[warn] {int(thin.sum())} slice(s) have < {MIN_VOX} brain voxels -- percentiles there "
          f"are noisy (cmap.yaml's drop_empty_slices/min_brain_frac already trims the worst)")

## The schemes

The `-noet` schemes differ from their twins in one line and one line only: which voxels the centre
and the scale are *estimated* over. The transform is applied to every voxel either way, so ET is
still present in the normalized image -- it just no longer votes on where zero and one sit.

In [ ]:
def et_fraction(brain, et, per_slice=PER_SLICE):
    '''-> (n,) ET fraction; constant across slices when per_slice is False.'''
    n = brain.shape[0]
    if per_slice:
        b = brain.reshape(n, -1).sum(1)
        return et.reshape(n, -1).sum(1) / np.maximum(b, 1)
    return np.full(n, et.sum() / max(brain.sum(), 1), np.float64)


def divisors(vol_c, region, p, per_slice=PER_SLICE):
    '''p-th percentile of one contrast over `region`. p is (n,). -> (n,) divisor per slice.
    per_slice=False pools every slice into one percentile and broadcasts it back.'''
    n = vol_c.shape[0]
    if not per_slice:
        v = vol_c[region]
        d = float(np.percentile(v, np.clip(float(p[0]), 0.0, 100.0))) if v.size else 1.0
        return np.full(n, d, np.float32)
    out = np.ones(n, np.float32)
    for z in range(n):
        v = vol_c[z][region[z]]
        if v.size:
            out[z] = np.percentile(v, np.clip(float(p[z]), 0.0, 100.0))
    return out


# (centre, scale, region) for the z-score family. `region` selects the voxels the statistics are
# ESTIMATED over; the transform is applied to the whole slice regardless.
#   "brain"   every brain voxel                              -- the original schemes
#   "noet"    brain & ~ET, for EVERY contrast                 -- match the unenhanced brain
#   "noetce"  brain & ~ET for T1ce only, brain elsewhere      -- only T1ce has the tail
# MAD and IQR carry the consistency constants that make them equal the std on Gaussian data, so
# every scheme below emits comparable units.
CENTRE_SCALE = {"zscore":            ("mean",   "std", "brain"),
                "median-std":        ("median", "std", "brain"),
                "median-mad":        ("median", "mad", "brain"),
                "median-iqr":        ("median", "iqr", "brain"),
                "zscore-noet":       ("mean",   "std", "noet"),
                "zscore-noetce":     ("mean",   "std", "noetce"),
                "median-std-noet":   ("median", "std", "noet"),
                "median-mad-noet":   ("median", "mad", "noet"),
                "median-iqr-noet":   ("median", "iqr", "noet"),
                "median-mad-noetce": ("median", "mad", "noetce")}


def stat_region(brain, et, region, c):
    '''Which voxels the centre/scale for stored contrast `c` is estimated over.'''
    if region == "brain":
        return brain
    if region == "noet":
        return brain & ~et
    if region == "noetce":
        return (brain & ~et) if c == T1CE else brain
    raise ValueError(f"unknown estimation region {region!r}")


def centre_scale(v, how):
    '''-> (centre, scale) of a 1-D sample under one of the CENTRE_SCALE recipes.'''
    c_how, s_how = how[0], how[1]
    c = float(v.mean()) if c_how == "mean" else float(np.median(v))
    if s_how == "std":
        s = float(v.std())
    elif s_how == "mad":
        s = 1.4826 * float(np.median(np.abs(v - np.median(v))))
    else:                                        # iqr
        s = float(np.percentile(v, 75) - np.percentile(v, 25)) / 1.349
    return c, max(s, 1e-8)


def fit_centre_scale(raw_c, region, how, brain, per_slice=PER_SLICE):
    '''(n,) centre and scale for one contrast, estimated over `region`.

    A region with fewer than MIN_STAT_VOX voxels falls back to the full brain mask -- otherwise a
    slice that is almost entirely ET (or a per-slice run on a tumor-edge slice) would set the
    scale from a handful of voxels. The fallback is counted and reported.'''
    n = raw_c.shape[0]
    cc = np.zeros(n, np.float32)
    ss = np.ones(n, np.float32)
    n_fallback = 0
    if not per_slice:
        v = raw_c[region]
        if v.size < MIN_STAT_VOX:
            v, n_fallback = raw_c[brain], 1
        if v.size:
            a, b = centre_scale(v, how)
            cc[:], ss[:] = a, b
        return cc, ss, n_fallback
    for z in range(n):
        v = raw_c[z][region[z]]
        if v.size < MIN_STAT_VOX:
            v = raw_c[z][brain[z]]
            n_fallback += 1
        if v.size:
            cc[z], ss[z] = centre_scale(v, how)
    return cc, ss, n_fallback


def normalize(raw, brain, et, scheme, p_base=P_BASE, per_slice=PER_SLICE):
    '''-> (normalized (n,H,W,4), info). Background stays 0.

    info['divisor'] is (n,4) -- the scale each contrast was divided by.
    info['centre']  is (n,4) -- the value subtracted (0 for the percentile family).'''
    n = raw.shape[0]
    out = np.zeros_like(raw)
    div = np.ones((n, 4), np.float32)
    ctr = np.zeros((n, 4), np.float32)
    p_t1ce = np.full(n, p_base, np.float64)
    n_fallback = 0

    if scheme in CENTRE_SCALE:                   # z-score family and its median / -noet variants
        how = CENTRE_SCALE[scheme]
        region_name = how[2]
        for c in range(4):
            reg = stat_region(brain, et, region_name, c)
            cc, ss, nf = fit_centre_scale(raw[..., c], reg, how, brain, per_slice)
            n_fallback += nf
            ctr[:, c], div[:, c] = cc, ss
            out[..., c] = (raw[..., c] - cc[:, None, None]) / ss[:, None, None]
        p_t1ce = None
    else:
        if scheme == "proposed":
            p_t1ce = np.maximum(p_base - 100.0 * et_fraction(brain, et, per_slice), P_FLOOR)
        for c in range(4):
            if scheme == "oracle" and c == T1CE:
                div[:, c] = divisors(raw[..., c], brain & ~et, np.full(n, p_base), per_slice)
            else:
                div[:, c] = divisors(raw[..., c], brain,
                                     p_t1ce if c == T1CE else np.full(n, p_base), per_slice)
        out = raw / np.maximum(div, 1e-8)[:, None, None, :]
    out = out * brain[..., None]
    return out, {"scheme": scheme, "divisor": div, "centre": ctr, "pct_t1ce": p_t1ce,
                 "n_fallback": n_fallback}


norm = {s: normalize(raw, brain, et, s) for s in SCHEMES}

nfb = {s: norm[s][1]["n_fallback"] for s in SCHEMES if norm[s][1]["n_fallback"]}
if nfb:
    print(f"[warn] estimation region below MIN_STAT_VOX={MIN_STAT_VOX}, fell back to full brain: "
          f"{nfb}\n")

zmax = int(np.argmax(et_z))          # the slice with the most tumor
print(f"centre / scale on slice {zmax} (the most-ET slice, f_ET = {f_et_z[zmax]:.3%}):\n")
print(f"{'scheme':<19} {'T1 ctr':>9} {'T1 scl':>9} {'T1ce ctr':>9} {'T1ce scl':>9} "
      f"{'scl ratio':>10} {'T1ce pct':>9}")
for s in SCHEMES:
    d, c_, p = norm[s][1]["divisor"], norm[s][1]["centre"], norm[s][1]["pct_t1ce"]
    pct = "-" if p is None else f"{p[zmax]:.3f}"
    print(f"{s:<19} {c_[zmax, T1]:>9.2f} {d[zmax, T1]:>9.2f} {c_[zmax, T1CE]:>9.2f} "
          f"{d[zmax, T1CE]:>9.2f} {d[zmax, T1CE] / max(d[zmax, T1], 1e-8):>10.3f} {pct:>9}")

if HAVE_PCT and has_et.any():
    r = (norm["proposed"][1]["divisor"][has_et, T1CE]
         / np.maximum(norm["oracle"][1]["divisor"][has_et, T1CE], 1e-8) - 1.0)
    print(f"\nproposed vs oracle T1ce divisor, over the {int(has_et.sum())} ET-bearing slices:")
    print(f"  median {np.median(r):+.2%}, IQR [{np.percentile(r, 25):+.2%}, "
          f"{np.percentile(r, 75):+.2%}], |err| < 5% on {np.mean(np.abs(r) < 0.05):.0%} of them")

## Does excluding ET move the estimator at all?

This is the first thing to check, and it is prior to every residual metric below: if dropping the ET
voxels barely changes the numbers that `median-mad` produces, then `median-mad-noet` is the same
normalization under a different name and nothing downstream can tell them apart.

There is a reason to expect exactly that. The MAD has a **breakdown point of 50%** -- it is
unaffected by arbitrary corruption of up to half the sample -- and ET is a few percent of the brain.
The median is the same. The **std has no breakdown point at all**: a tail of $f$ voxels sitting
$\delta$ above the bulk inflates the variance by roughly $f\delta^2$ regardless of how small $f$ is,
so it responds immediately.

Below, each recipe is fitted twice on the same slices -- once over the brain, once over
brain $\setminus$ ET -- and the two are compared directly. The centre shift is reported in units of
the brain-wide scale, so it is on the same footing as the `bias` column later.

Read it as a ranking of *how much the mask can possibly do*. A recipe whose numbers are near zero
here cannot behave differently downstream no matter what the residual tables say; a recipe with a
large shift is one where `-noet` is a genuinely different normalization. `mean-std` should sit at the
top of that ranking and `median-mad` at the bottom.

In [ ]:
RECIPES = [("median", "mad"), ("median", "std"), ("mean", "std")]
SHIFT_C = T1CE          # the contrast with the tail; set to T1 to confirm the null case


def estimator_shift(raw, brain, et, c=SHIFT_C, per_slice=PER_SLICE):
    '''-> {recipe: (dcentre_over_scale, dscale_rel)} per slice, brain vs brain&~ET.'''
    out = {}
    for how in RECIPES:
        cb, sb, _ = fit_centre_scale(raw[..., c], brain, how, brain, per_slice)
        cn, sn, _ = fit_centre_scale(raw[..., c], brain & ~et, how, brain, per_slice)
        out[f"{how[0]}-{how[1]}"] = ((cn - cb) / np.maximum(sb, 1e-8), sn / np.maximum(sb, 1e-8) - 1)
    return out


shift = estimator_shift(raw, brain, et)
sel_s = has_et if has_et.any() else np.ones(n, bool)

print(f"{CONTRASTS[SHIFT_C]}: brain&~ET vs brain, over the {int(sel_s.sum())} ET-bearing "
      f"{'slices' if PER_SLICE else 'slices of this volume (stats are volume-level)'}\n")
print(f"{'recipe':<12} {'d centre / scale':>28} {'d scale (relative)':>28}")
for k, (dc, ds) in shift.items():
    a = np.percentile(dc[sel_s], [25, 50, 75])
    b = np.percentile(ds[sel_s], [25, 50, 75])
    print(f"{k:<12} {a[1]:>+11.4f} [{a[0]:+.4f}, {a[2]:+.4f}] "
          f"{b[1]:>+11.2%} [{b[0]:+.2%}, {b[2]:+.2%}]")
print("\nIf median-mad's numbers are ~0 here, median-mad-noet cannot differ from median-mad "
      "downstream:\nthe robust estimators already ignored the tail, mask or no mask. mean-std is "
      "the opposite case --\nno breakdown point, so the mask is doing real work there, and "
      "zscore-noet is where to look for a win.")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for k, (dc, ds) in shift.items():
    ax[0].scatter(100 * f_et_z[sel_s], dc[sel_s], s=14, alpha=0.7, label=k)
    ax[1].scatter(100 * f_et_z[sel_s], 100 * ds[sel_s], s=14, alpha=0.7, label=k)
ax[0].set_ylabel("centre shift / scale"); ax[1].set_ylabel("scale change (%)")
ax[0].set_title(f"{CONTRASTS[SHIFT_C]} centre: dropping ET", fontsize=9)
ax[1].set_title(f"{CONTRASTS[SHIFT_C]} scale: dropping ET", fontsize=9)
for a in ax:
    a.axhline(0, c="k", lw=0.8); a.grid(alpha=0.3); a.legend(fontsize=8)
    a.set_xlabel(r"per-slice $f_{ET}$ (%)")
plt.tight_layout(); plt.show()

## Scale and centre along the slice axis

Where the schemes separate is where the estimation region is doing work. On ET-free slices every
`-noet` scheme is *identical by construction* to its brain-wide twin, so the grey bands below carry
no information -- read the white ones.

(With `PER_SLICE = False` these are flat lines: the whole volume shares one estimate. That is the
point of the panel in that mode -- it shows the size of the offset between schemes, not its
variation.)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14.5, 3.8))
zz = np.arange(n)

for s in SCHEMES:
    ax[0].plot(zz, norm[s][1]["divisor"][:, T1CE], lw=1.2, label=s)
ax[0].set_title("T1ce scale (divisor) per slice", fontsize=10)
ax[0].set_xlabel("slice"); ax[0].legend(fontsize=7)

plotted = 0   # PAIRS comes from the setup cell
for a_, b_ in PAIRS:
    if a_ in norm and b_ in norm:
        ax[1].plot(zz, norm[a_][1]["divisor"][:, T1CE]
                   / np.maximum(norm[b_][1]["divisor"][:, T1CE], 1e-8) - 1.0,
                   lw=1.3, label=f"{a_} / {b_}")
        plotted += 1
ax[1].axhline(0, c="k", lw=0.8)
ax[1].set_title("T1ce scale: -noet relative to its twin", fontsize=10)
ax[1].set_xlabel("slice"); ax[1].set_ylabel("relative change")
if plotted:
    ax[1].legend(fontsize=7)

ax[2].plot(zz, 100 * f_et_z, lw=1.4, c="crimson")
ax[2].set_title(r"$100 f_{ET}$ per slice", fontsize=10)
ax[2].set_xlabel("slice"); ax[2].set_ylabel("% of brain")

for a in ax:
    a.grid(alpha=0.3)
    for z0 in zz[~has_et]:
        a.axvspan(z0 - 0.5, z0 + 0.5, color="0.9", zorder=0)
ax[0].text(0.02, 0.03, "grey = slices with no ET", transform=ax[0].transAxes, fontsize=7)
plt.tight_layout(); plt.show()

if HAVE_PCT:
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
    for s in PCT:
        ax[0].plot(zz, norm[s][1]["divisor"][:, T1CE], lw=1.4, label=s)
    ax[0].plot(zz, norm["p99"][1]["divisor"][:, T1], lw=1.0, ls="--", c="k", label="T1 (p99)")
    ax[0].set_title("percentile family: T1ce divisor", fontsize=10)
    ax[0].set_xlabel("slice"); ax[0].legend(fontsize=8)
    ax[1].plot(zz, norm["proposed"][1]["divisor"][:, T1CE]
               / np.maximum(norm["oracle"][1]["divisor"][:, T1CE], 1e-8) - 1.0, lw=1.4)
    ax[1].axhline(0, c="k", lw=0.8)
    ax[1].set_title("proposed / oracle - 1   (approximation error)", fontsize=10)
    ax[1].set_xlabel("slice")
    for a in ax:
        a.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

## Does $\Delta = $ T1ce $-$ T1 isolate the tumor?

Over non-ET brain we want $\Delta$ **centred on zero** (no global offset) and **tight** (no anatomy
leaking in); inside ET we want it large. `separation` is the ET median over the non-ET RMS.

**The `-noet` schemes are fitted on exactly the region `bias` and `bg_rms` are measured over**, so
those two columns flatter them for definitional reasons and should not be used to rank.

`separation` and `leak` are not a way around that. `separation` is `et_med / bg_rms` and `leak`'s
threshold is `0.5 * et_med`, so both carry a fit target in the denominator: a scheme that flattens
`bg_rms` inflates `separation` for free. The two clean columns are **`et_med`** (ET voxels only,
divided by a brain-wide reference every scheme sets) and **`sigma`**.

Every ratio-free spread column is divided by the normalized-T1 std $s$, so schemes that define their
own units stay comparable. `sigma` is deliberately **not** divided by anything: it is reported in
the scheme's own units because that is the scale the data would actually be stored at.

Reported twice. **all slices** is what the trained network would see. **ET slices only** is the
discriminating set -- ET-free slices are identical across `-noet` schemes and their twins by
construction, so including them can only pull the schemes together.

In [ ]:
def residual_stats(x, brain, et, keep=None):
    '''keep: optional (n,) bool selecting slices to pool over.'''
    if keep is not None:
        x, brain, et = x[keep], brain[keep], et[keep]
    d = x[..., T1CE] - x[..., T1]
    bg, tu = d[brain & ~et], d[et]
    if bg.size == 0:
        return {k: float("nan") for k in
                ("bias", "bg_rms", "bg_iqr", "et_med", "separation", "leak", "sigma", "sigma_dl")}
    # sigma = RMS(T1ce - T1) over the WHOLE brain in the scheme's own units. This is the varsigma
    # that centres an I2SB schedule (std_fwd[-1] == varsigma); sigma_dl is the same after the
    # dataloader's per-contrast divisor, i.e. what the bridge would actually be sized against.
    sig = float(np.sqrt(np.mean(d[brain] ** 2)))
    # Each scheme defines its own units -- std units for the z-score family, fraction-of-p99 for
    # the percentile family -- so RAW bias and RMS are not comparable across them. Divide by the
    # spread of the normalized T1, which every scheme also sets, and the columns mean the same
    # thing everywhere. `separation` and `leak` are ratios, so they come out unchanged.
    ref = max(float(np.std(x[..., T1][brain])), 1e-8)
    bg, tu = bg / ref, tu / ref
    s = {"bias": float(np.median(bg)),                  # want ~0
         "bg_rms": float(np.sqrt(np.mean(bg ** 2))),    # want small
         "bg_iqr": float(np.percentile(bg, 75) - np.percentile(bg, 25)),
         "et_med": float(np.median(tu)) if tu.size else float("nan"),
         "sigma": sig,
         "sigma_dl": sig / DIV_SCALE}
    s["separation"] = s["et_med"] / max(s["bg_rms"], 1e-8)
    s["leak"] = float(np.mean(bg > 0.5 * s["et_med"])) if tu.size else float("nan")
    return s


# Clean = a fit target of the -noet schemes appears in NEITHER the numerator nor the denominator.
# separation (= et_med / bg_rms) and leak (threshold 0.5*et_med) both divide by one, so they are
# reported but not ranked on -- a scheme that flattens bg_rms inflates them for free.
CLEAN = {"et_med", "sigma", "sigma_dl"}

for label, keep in [("all slices", None), ("ET slices only", has_et)]:
    rows_ = {s: residual_stats(norm[s][0], brain, et, keep) for s in SCHEMES}
    nsl = n if keep is None else int(keep.sum())
    print(f"--- {label}  ({nsl} slices) ---")
    print(f"{'scheme':<19} {'bias/s':>9} {'rms/s':>8} {'iqr/s':>8} | {'etmed/s':>8} "
          f"{'sigma':>8} | {'separatn':>10} {'leak':>7}")
    print(f"{'':<19} {'<--- fit target of -noet --->':^28} | {'<-- clean -->':^17} | "
          f"{'<-- mixed -->':^18}")
    for s in SCHEMES:
        r = rows_[s]
        print(f"{s:<19} {r['bias']:>+9.4f} {r['bg_rms']:>8.4f} {r['bg_iqr']:>8.4f} | "
              f"{r['et_med']:>8.4f} {r['sigma']:>8.4f} | {r['separation']:>10.4g} "
              f"{r['leak']:>7.2%}")
    print()
    if keep is has_et:
        rows = rows_          # keep the discriminating set for the figures below
print("Rank on etmed and sigma. bias/bg_rms are the -noet fit targets, and separation/leak\n"
      "divide by one -- flattening bg_rms inflates them without the tumor contrast moving.")

## What each scheme means for the I2SB schedule

`sigma` above is $\varsigma = \mathrm{RMS}(x_0 - x_1)$ over the brain, and the schedule-centring
criterion in `inspect_beta_max.ipynb` is `std_fwd[-1] == `$\varsigma$ -- so the normalization
directly sets `beta_max`. Two things to watch:

* **The `linear_start` floor.** `sb/base.py`'s `i2sb_betas` hardcodes `linear_start = 1e-4`, which
  pins `std_fwd[-1] >= 0.2416` no matter how small `beta_max` goes. A scheme whose $\varsigma$ (after
  the dataloader's `scales` divisor, `DIV_SCALE` here) lands **below 0.2416** cannot be centred by
  the faithful i2sb schedule at all; it needs either a smaller `linear_start` or the brownian
  schedule, whose `tau` is a free scale with no floor.
* **$\varsigma$ can move either way, and only the table settles it.** Excluding ET pulls two levers
  in opposite directions: the healthy residual gets flatter, pushing $\varsigma$ down, but the scale
  estimate also shrinks -- the ET tail was inflating it -- and dividing by a smaller number magnifies
  everything, the ET spike included, pushing $\varsigma$ back up. Read `beta_max_hat` off the table
  rather than reasoning about which way it went.

`beta_max_hat` below is solved numerically from the same schedule the training code builds, so it is
the number to paste into **both** `cfg["i2sb"]["beta_max"]` and `cfg["model"]["params"]["beta_max"]`.

In [ ]:
def i2sb_std_fwd_last(beta_max, n_pts=1000, linear_start=1e-4):
    '''std_fwd[-1] for sb/base.py's i2sb_betas -- the total forward std of the bridge.'''
    b = np.linspace(linear_start ** 0.5, (beta_max / n_pts) ** 0.5, n_pts, dtype=np.float64) ** 2
    b = np.concatenate([b[: n_pts // 2], np.flip(b[: n_pts // 2])])
    return float(np.sqrt(b.sum()))


FLOOR = i2sb_std_fwd_last(1e-12)
print(f"i2sb linear_start=1e-4 floor: std_fwd[-1] >= {FLOOR:.4f}  (beta_max cannot go below it)\n")


def solve_beta_max(target, lo=1e-4, hi=50.0, iters=80):
    if target <= i2sb_std_fwd_last(lo):
        return float("nan")
    for _ in range(iters):
        mid = 0.5 * (lo + hi)
        if i2sb_std_fwd_last(mid) < target:
            lo = mid
        else:
            hi = mid
    return 0.5 * (lo + hi)


lab_sig = f"sigma/{DIV_SCALE:g}"
print(f"{'scheme':<19} {'sigma':>8} {lab_sig:>10} "
      f"{'beta_max_hat':>13} {'brownian tau':>13}")
for s in SCHEMES:
    sig = rows[s]["sigma"]           # ET-slice pool, the discriminating set
    sig_dl = sig / DIV_SCALE
    bm = solve_beta_max(sig_dl)
    flag = "" if np.isfinite(bm) else "   <- BELOW THE FLOOR"
    bms = f"{bm:>13.4f}" if np.isfinite(bm) else f"{'--':>13}"
    print(f"{s:<19} {sig:>8.4f} {sig_dl:>10.4f} {bms} {sig_dl / 2:>13.4f}{flag}")
print(f"\nbrownian tau = sigma/2 (std_fwd(t) = 2 tau sqrt(t), so std_fwd(1) = 2 tau).")
print(f"'BELOW THE FLOOR' means the faithful i2sb schedule cannot be centred on that sigma: use\n"
      f"the brownian schedule, or lower linear_start in sb/base.py:i2sb_betas.")

## Images

The slice with the most ET. Bottom row is $\Delta$ on a symmetric diverging scale (white = 0) with
the ET boundary drawn on -- under the hypothesis it should be flat white outside the contour.
Normalization here is slice-local, so these values are set entirely by this slice's own statistics.

In [ ]:
z = zmax
print(f"slice {z}: ET = {et_z[z]:,} px ({f_et_z[z]:.2%} of this slice's brain)")

ncol = len(SCHEMES) + 1
fig, ax = plt.subplots(2, ncol, figsize=(3.0 * ncol, 6.2))
ax[0][0].imshow(raw[z, ..., T1] * brain[z], cmap="gray")
ax[0][0].set_title("T1 (raw)", fontsize=9)
ax[1][0].imshow(raw[z, ..., T1CE] * brain[z], cmap="gray")
ax[1][0].set_title("T1ce (raw)", fontsize=9)
for a in (ax[0][0], ax[1][0]):
    a.contour(et[z], levels=[0.5], colors="r", linewidths=0.6)

for j, s in enumerate(SCHEMES, start=1):
    x = norm[s][0]
    lo, hi = np.percentile(x[z, ..., T1CE][brain[z]], [1, 99])
    ax[0][j].imshow(x[z, ..., T1CE], cmap="gray", vmin=lo, vmax=hi)
    ax[0][j].set_title(f"{s}\nT1ce normalized", fontsize=9)
    d = (x[z, ..., T1CE] - x[z, ..., T1]) * brain[z]
    v = float(np.percentile(np.abs(d[brain[z]]), 99)) or 1.0
    im = ax[1][j].imshow(d, cmap="bwr", vmin=-v, vmax=v)
    ax[1][j].contour(et[z], levels=[0.5], colors="k", linewidths=0.6)
    sep_z = residual_stats(x[z:z + 1], brain[z:z + 1], et[z:z + 1])["separation"]
    ax[1][j].set_title(f"$\\Delta$   sep(this slice)={sep_z:.2f}", fontsize=9)
    plt.colorbar(im, ax=ax[1][j], fraction=0.046)
for a in ax.ravel():
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## Histograms

Pooled over **ET-bearing slices only**, for the reason above. Left: T1 vs T1ce over the brain -- the
hypothesis says these should overlap once T1ce's tumor tail is stepped over. Right: the residual,
split healthy / ET. Under the hypothesis the healthy curve is a narrow spike at 0 and the ET curve
sits clearly to its right.

In [ ]:
sel = has_et if has_et.any() else np.ones(n, bool)
fig, ax = plt.subplots(2, len(SCHEMES), figsize=(3.3 * len(SCHEMES), 6.0))
for j, s in enumerate(SCHEMES):
    x, b_, e_ = norm[s][0][sel], brain[sel], et[sel]
    t1, t1ce = x[..., T1][b_], x[..., T1CE][b_]
    lo, hi = np.percentile(np.concatenate([t1, t1ce]), [0.5, 99.5])
    bins = np.linspace(lo, hi, 120)
    ax[0][j].hist(t1, bins=bins, histtype="step", density=True, label="T1")
    ax[0][j].hist(t1ce, bins=bins, histtype="step", density=True, label="T1ce")
    ax[0][j].set_title(s, fontsize=10); ax[0][j].set_yscale("log")
    if j == 0:
        ax[0][j].legend(fontsize=8); ax[0][j].set_ylabel("brain, log density")

    d = x[..., T1CE] - x[..., T1]
    bg, tu = d[b_ & ~e_], d[e_]
    b2 = np.linspace(*np.percentile(d[b_], [0.5, 99.5]), 120)
    ax[1][j].hist(bg, bins=b2, histtype="step", density=True, label="healthy")
    if tu.size:
        ax[1][j].hist(tu, bins=b2, histtype="step", density=True, label="ET")
    ax[1][j].axvline(0, color="k", lw=0.8); ax[1][j].set_yscale("log")
    ax[1][j].set_xlabel(r"$\Delta$ = T1ce - T1")
    if j == 0:
        ax[1][j].legend(fontsize=8); ax[1][j].set_ylabel("log density")
plt.tight_layout(); plt.show()

## The two families side by side, across subjects

`SHOW_SCHEMES` on the most-ET slice of `N_SHOW` random subjects. The default pairs each family with
its ET-masked version -- `zscore` / `zscore-noet` against `median-mad` / `median-mad-noet` -- so the
row reads as: does the mask move the mean into the territory the median already occupied?

Under the hypothesis, $\Delta = $ T1ce $-$ T1 should be **flat white outside the ET contour and hot
inside it**.

### Everything here is in absolute i2sb units

Every image is divided by `DIV_SCALE` (the `scales` divisor `i2sb_dataset` applies, 3.0) and
displayed on a **fixed $[-1, +1]$ window** -- your bridge convention, `data_range: 2.0`. So these
are literally the arrays the network receives, and a panel that saturates is a panel whose data
does not fit the bridge's range. The `%out` column below counts exactly that.

All the $\Delta$ maps share **one** window, computed once over every subject and every scheme, and
that window is taken from the **healthy** residual -- the ET spike is one to two orders of magnitude
larger, so a brain-wide percentile would be set by the tumor and render the background blank white.
ET therefore saturates on purpose; its size is in the histograms and the table.

No per-scheme rescaling is applied. The schemes genuinely produce different scales -- `-noet`
estimates its spread without the tumor tail, so its divisor is smaller and every value it produces
is larger -- and on an absolute axis you see that difference instead of having it divided out. That
is the right view for "what does the data look like", but it means you **cannot** read the $\Delta$
panels as a quality ranking: a scheme can look flatter purely by being scaled down. The residual
tables above put the schemes in common units for that purpose; these pictures are the deployment
view.

In [ ]:
N_SHOW       = 4                        # subjects in the grid
SHOW_SCHEMES = ["zscore", "zscore-noet", "median-mad", "median-mad-noet"]
IMG_VLIM     = 1.0                      # i2sb convention: show images on [-1, +1] after /DIV_SCALE
DELTA_VLIM   = None                     # None -> one shared window from the pooled healthy residual
MAX_HIST_VOX = 200_000                  # healthy voxels kept per subject per scheme

assert all(s in CENTRE_SCALE or s in ("p99", "proposed", "oracle") for s in SHOW_SCHEMES)

rng_v = np.random.default_rng(SEED)
panels = []
acc = {s: {"bg": [], "et": [], "t1ce": [], "sq": [0.0, 0], "oor": [0, 0, 0]} for s in SHOW_SCHEMES}

for si in rng_v.permutation(len(usable)):
    if len(panels) >= N_SHOW:
        break
    p = usable[int(si)]
    try:
        r_, b_, e_ = load_subject(p)
    except Exception as err:
        print(f"[skip] {os.path.basename(p)}: {type(err).__name__}")
        continue
    nz = r_.shape[0]
    ez = e_.reshape(nz, -1).sum(1)
    keep = ez > 0
    if not keep.any():
        continue
    z = int(np.argmax(ez))                          # the slice with the most tumor
    rec = {"subject": os.path.basename(os.path.dirname(p)), "z": z,
           "f_et": float(ez[z] / max(b_[z].sum(), 1)),
           "t1_raw": r_[z, ..., T1] * b_[z], "brain": b_[z], "et": e_[z] > 0.5}

    for s in SHOW_SCHEMES:
        # ABSOLUTE units: exactly what i2sb_dataset hands the network (img / scales). No
        # per-scheme rescaling -- the schemes really do produce different scales, and that
        # difference is the thing this view is meant to show.
        x_ = normalize(r_, b_, e_, s)[0] / DIV_SCALE
        d_ = x_[..., T1CE] - x_[..., T1]
        rec[s] = {"t1ce": x_[z, ..., T1CE], "d": d_[z] * b_[z]}

        bk, ek = b_[keep], e_[keep]
        bg, tu = d_[keep][bk & ~ek], d_[keep][ek]
        acc[s]["bg"].append(rng_v.choice(bg, MAX_HIST_VOX, replace=False)
                            if bg.size > MAX_HIST_VOX else bg)
        acc[s]["et"].append(tu)
        ce = x_[keep][..., T1CE][bk]
        acc[s]["t1ce"].append(rng_v.choice(ce, MAX_HIST_VOX, replace=False)
                              if ce.size > MAX_HIST_VOX else ce)
        acc[s]["sq"][0] += float(np.sum(d_[keep][bk] ** 2)); acc[s]["sq"][1] += int(bk.sum())
        acc[s]["oor"][0] += int(np.sum(np.abs(x_[keep][..., T1][bk]) > IMG_VLIM))
        acc[s]["oor"][1] += int(np.sum(np.abs(ce) > IMG_VLIM))
        acc[s]["oor"][2] += int(bk.sum())
    panels.append(rec)
    print(f"  {len(panels)}/{N_SHOW}  {rec['subject']}  slice {z}, "
          f"f_ET={rec['f_et']:.2%}      ", end="\r")

if not panels:
    raise RuntimeError("no subject with ET voxels was loaded")
print(f"\n{len(panels)} subject(s) shown, all values in i2sb units (normalized / {DIV_SCALE:g})\n")

# ---- ONE delta window for every panel, from the healthy residual pooled over all schemes
vd = DELTA_VLIM or float(np.percentile(np.concatenate(
    [np.abs(r[s]["d"])[r["brain"] & ~r["et"]].ravel() for r in panels for s in SHOW_SCHEMES]),
    99.5)) or 1.0

ncol = 1 + 2 * len(SHOW_SCHEMES)
cols = ["T1 (raw, own window)"]
for s in SHOW_SCHEMES:
    cols += [f"T1ce  {s}\n[$\\pm${IMG_VLIM:g}]", f"$\\Delta$  {s}\n[$\\pm${vd:.3f}]"]

fig, ax = plt.subplots(len(panels), ncol, figsize=(2.1 * ncol, 2.35 * len(panels)), squeeze=False)
for i, r in enumerate(panels):
    imgs = [(r["t1_raw"], "gray", None, None)]
    for s in SHOW_SCHEMES:
        # background 0 sits mid-window on a signed scale; park it at the floor
        imgs.append((np.where(r["brain"], r[s]["t1ce"], -IMG_VLIM), "gray", -IMG_VLIM, IMG_VLIM))
        imgs.append((r[s]["d"], "bwr", -vd, vd))
    for j, (im_, cm, lo_, hi_) in enumerate(imgs):
        ax[i][j].imshow(im_, cmap=cm, vmin=lo_, vmax=hi_)
        ax[i][j].contour(r["et"], levels=[0.5], colors="k" if j else "r", linewidths=0.6)
        ax[i][j].set_xticks([]); ax[i][j].set_yticks([])
        if i == 0:
            ax[i][j].set_title(cols[j], fontsize=7.5)
    ax[i][0].set_ylabel(f"{r['subject']}\n$f_{{ET}}$={r['f_et']:.1%}", fontsize=7.5)
fig.suptitle(f"absolute i2sb units (normalized / {DIV_SCALE:g}); images on a fixed "
             f"[-{IMG_VLIM:g}, +{IMG_VLIM:g}], one shared $\\Delta$ window "
             f"[$\\pm${vd:.3f}] set by the HEALTHY residual so ET saturates by design",
             fontsize=8.5, y=1.004)
plt.tight_layout(); plt.show()

# ---- pooled distributions over every ET-bearing slice of the shown subjects
bg_all = {s: np.concatenate(acc[s]["bg"]) for s in SHOW_SCHEMES}
et_all = {s: np.concatenate(acc[s]["et"]) for s in SHOW_SCHEMES}
ce_all = {s: np.concatenate(acc[s]["t1ce"]) for s in SHOW_SCHEMES}

fig, ax = plt.subplots(1, 3, figsize=(16.5, 4))
b1 = np.linspace(*np.percentile(np.concatenate(list(bg_all.values())), [0.2, 99.8]), 160)
b2 = np.linspace(b1[0], float(np.percentile(np.concatenate(list(et_all.values())), 99.5)), 200)
b3 = np.linspace(*np.percentile(np.concatenate(list(ce_all.values())), [0.1, 99.9]), 200)
for s in SHOW_SCHEMES:
    ax[0].hist(bg_all[s], bins=b1, histtype="step", density=True, lw=1.4, label=s)
    ax[1].hist(et_all[s], bins=b2, histtype="step", density=True, lw=1.4, label=s)
    ax[2].hist(ce_all[s], bins=b3, histtype="step", density=True, lw=1.4, label=s)
ax[0].set_title(r"healthy brain: $\Delta$ should be a narrow spike at 0", fontsize=9)
ax[1].set_title(r"ET: $\Delta$ should sit far to the right", fontsize=9)
ax[2].set_title(f"normalized T1ce -- does it fit [-{IMG_VLIM:g}, +{IMG_VLIM:g}]?", fontsize=9)
ax[2].axvspan(-IMG_VLIM, IMG_VLIM, color="0.85", zorder=0)
for a in ax[:2]:
    a.set_xlabel(rf"$\Delta$ = T1ce - T1   (normalized / {DIV_SCALE:g})")
ax[2].set_xlabel(f"T1ce  (normalized / {DIV_SCALE:g});  grey = the i2sb range")
for a in ax:
    a.axvline(0, c="k", lw=0.8); a.grid(alpha=0.3); a.legend(fontsize=8)
ax[0].set_ylabel("density")
plt.tight_layout(); plt.show()

# ---- the numbers, all in absolute i2sb units
print(f"pooled over every ET-bearing slice of the {len(panels)} subject(s) shown; "
      f"i2sb units (normalized / {DIV_SCALE:g})\n")
print(f"{'scheme':<19} {'hlth med':>9} {'hlth RMS':>9} {'ET med':>8} {'sigma':>8} | "
      f"{'%T1 out':>8} {'%T1ce out':>10}")
summ = {}
for s in SHOW_SCHEMES:
    bg, tu = bg_all[s], et_all[s]
    sig = (acc[s]["sq"][0] / max(acc[s]["sq"][1], 1)) ** 0.5
    o1, oc, ot = acc[s]["oor"]
    summ[s] = {"hmed": float(np.median(bg)), "hrms": float(np.sqrt(np.mean(bg ** 2))),
               "etmed": float(np.median(tu)) if tu.size else float("nan"), "sigma": sig,
               "o1": o1 / max(ot, 1), "oce": oc / max(ot, 1)}
    v = summ[s]
    print(f"{s:<19} {v['hmed']:>+9.4f} {v['hrms']:>9.4f} {v['etmed']:>8.3f} {v['sigma']:>8.4f} | "
          f"{v['o1']:>8.2%} {v['oce']:>10.2%}")
print(f"\n%out = brain voxels outside [-{IMG_VLIM:g}, +{IMG_VLIM:g}] after /{DIV_SCALE:g}: what the "
      f"bridge's data_range does not cover.")
print("sigma feeds beta_max -- cross-check it against the schedule table above (which is pooled "
      "over\nmore subjects). These columns are ABSOLUTE, so a smaller number can just mean a "
      "smaller divisor;\nuse the residual tables, which divide it out, to rank the schemes.")

for a_, b_ in PAIRS:
    if a_ in summ and b_ in summ:
        print(f"\n{b_}  ->  {a_}   (does the mask move this estimator?)")
        for k, lab in [("hmed", "healthy median"), ("hrms", "healthy RMS"),
                       ("etmed", "ET median"), ("sigma", "sigma")]:
            print(f"  {lab:<16} {summ[b_][k]:>9.4f} -> {summ[a_][k]:>9.4f}   "
                  f"({summ[a_][k] / (summ[b_][k] if abs(summ[b_][k]) > 1e-9 else np.nan) - 1:+.1%})")

## Robustness across subjects

One subject proves nothing -- per-slice $f_{ET}$ varies enormously, and so does how well
$99 - 100 f_{ET}$ tracks the oracle. This resamples `N_SUBJECTS` at random. All statistics are
pooled over **ET-bearing slices** of each subject.

In [ ]:
N_SUBJECTS = 20

rng = np.random.default_rng(SEED)
pick = rng.choice(len(usable), size=min(N_SUBJECTS, len(usable)), replace=False)

recs, slice_err, shift_acc = [], [], {f"{a}-{b}": [] for a, b in RECIPES}
for i, si in enumerate(pick):
    p = usable[int(si)]
    try:
        r_, b_, e_ = load_subject(p)
    except Exception as err:
        print(f"[skip] {os.path.basename(p)}: {type(err).__name__}")
        continue
    nz = r_.shape[0]
    ez = e_.reshape(nz, -1).sum(1)
    keep = ez > 0
    if not keep.any():
        print(f"[skip] {os.path.basename(os.path.dirname(p))}: no ET voxels")
        continue
    rec = {"subject": os.path.basename(os.path.dirname(p)),
           "n_et_slices": int(keep.sum()), "n_slices": nz,
           "f_et": float(np.median((ez / np.maximum(b_.reshape(nz, -1).sum(1), 1))[keep])),
           "div": {}}
    for s in SCHEMES:
        x_, i_ = normalize(r_, b_, e_, s)
        rec[s] = residual_stats(x_, b_, e_, keep)
        rec["div"][s] = i_["divisor"][keep, T1CE]
    for k, (dc, ds) in estimator_shift(r_, b_, e_).items():
        shift_acc[k].append(np.stack([dc[keep], ds[keep]], 1))
    if HAVE_PCT:
        slice_err.append(rec["div"]["proposed"] / np.maximum(rec["div"]["oracle"], 1e-8) - 1.0)
    recs.append(rec)
    print(f"  {i + 1}/{len(pick)} {rec['subject']}  "
          f"{rec['n_et_slices']}/{nz} ET slices, median f_ET={rec['f_et']:.2%}      ", end="\r")

print(f"\n\n{len(recs)} subject(s) with ET; "
      f"{sum(r['n_et_slices'] for r in recs)} ET-bearing slices in total\n")
print(f"{'scheme':<19} {'|bias| med (IQR)':>22} {'bg_rms med (IQR)':>22} "
      f"{'etmed':>8} {'sigma':>8} {'separatn':>10} {'leak':>7}")
print(f"{'':<19} {'<------- fit target of -noet ------->':^45} "
      f"{'<-- clean -->':^17} {'<- mixed ->':^18}")
for s in SCHEMES:
    q = lambda k: np.nanpercentile([abs(r[s][k]) if k == "bias" else r[s][k] for r in recs],
                                   [25, 50, 75])
    b, g, sep, lk = q("bias"), q("bg_rms"), q("separation"), q("leak")
    sg, em = q("sigma"), q("et_med")
    print(f"{s:<19} {b[1]:>9.4f} ({b[0]:.3f}-{b[2]:.3f}) {g[1]:>9.4f} ({g[0]:.3f}-{g[2]:.3f}) "
          f"{em[1]:>8.4f} {sg[1]:>8.4f} {sep[1]:>10.4g} {lk[1]:>7.2%}")

print(f"\nestimator shift for {CONTRASTS[SHIFT_C]} (brain&~ET vs brain), pooled over all "
      f"ET-bearing slices:")
print(f"{'recipe':<12} {'|d centre| / scale  med (p90)':>34} {'|d scale| relative  med (p90)':>34}")
for k, v in shift_acc.items():
    if not v:
        continue
    a = np.concatenate(v)
    dc, ds = np.abs(a[:, 0]), np.abs(a[:, 1])
    print(f"{k:<12} {np.median(dc):>18.4f} ({np.percentile(dc, 90):.4f}) "
          f"{np.median(ds):>18.2%} ({np.percentile(ds, 90):.2%})")
print("  ^ the decisive number: if median-mad's shift is negligible here, -noet is a no-op.")

if HAVE_PCT:
    allerr = np.concatenate(slice_err)
    print(f"\nproposed / oracle T1ce divisor, PER SLICE over {allerr.size} ET-bearing slices:")
    print(f"  median {np.median(allerr):+.2%}, IQR [{np.percentile(allerr, 25):+.2%}, "
          f"{np.percentile(allerr, 75):+.2%}], |err| < 5% on {np.mean(np.abs(allerr) < 0.05):.0%}")

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(17, 4.0))
xs = np.arange(len(SCHEMES))
for a, key, ttl in zip(ax, ["et_med", "sigma", "separation", "bg_rms"],
                       ["et_med -- CLEAN (ET voxels, never fitted)",
                        r"$\varsigma$ = RMS(T1ce-T1) -- CLEAN, sets beta_max",
                        "separation -- MIXED (divides by bg_rms)",
                        "non-ET RMS -- FIT TARGET of -noet"]):
    for k, s in enumerate(SCHEMES):
        v = [r[s][key] for r in recs]
        a.scatter(np.full(len(v), k) + rng.normal(0, 0.06, len(v)), v, s=12, alpha=0.6)
        a.scatter([k], [np.nanmedian(v)], marker="_", s=600, c="k", zorder=3)
    a.set_xticks(xs); a.set_xticklabels(SCHEMES, rotation=35, ha="right", fontsize=7)
    a.set_title(ttl, fontsize=8); a.grid(alpha=0.3)
ax[1].axhline(FLOOR * DIV_SCALE, c="crimson", ls="--", lw=1.0)
ax[1].text(0.02, 0.02, f"i2sb floor ({DIV_SCALE:.0f}x{FLOOR:.3f})", transform=ax[1].transAxes,
           fontsize=7, color="crimson")
ax[2].set_yscale("log")     # -noet can drive bg_rms toward 0, which sends separation to infinity
plt.tight_layout(); plt.show()

# does the ET correction help most where the tail is biggest? one panel per -noet/twin pair
pairs = [(a_, b_) for a_, b_ in PAIRS if a_ in SCHEMES and b_ in SCHEMES]
if pairs:
    fig, ax = plt.subplots(1, len(pairs), figsize=(4.6 * len(pairs), 3.6), squeeze=False)
    for j, (a_, b_) in enumerate(pairs):
        ax[0][j].scatter([r["f_et"] for r in recs],
                         [r[a_]["et_med"] - r[b_]["et_med"] for r in recs], s=18)
        ax[0][j].axhline(0, color="k", lw=0.8); ax[0][j].grid(alpha=0.3)
        ax[0][j].set_xlabel(r"median per-slice $f_{ET}$")
        ax[0][j].set_ylabel("et_med gain (clean)")
        ax[0][j].set_title(f"{a_} - {b_}", fontsize=9)
    plt.tight_layout(); plt.show()

if HAVE_PCT:
    fig, a = plt.subplots(figsize=(5.5, 3.6))
    a.hist(allerr, bins=60); a.axvline(0, color="k", lw=0.8); a.grid(alpha=0.3)
    a.set_xlabel("proposed / oracle - 1, per ET-bearing slice")
    a.set_title("does the percentile rule land on the healthy-tissue p99?", fontsize=9)
    plt.tight_layout(); plt.show()

## Where the normalized data actually lands

The distribution each scheme produces over brain voxels, for one subject and for a batch. Linear
y-axis -- a log axis flatters the tails and hides how much mass sits near 0, which is the part that
sets a sensible data range downstream.

Set `SCHEME_SHOW` to compare two: the pair `median-mad` / `median-mad-noet` is the one in question.

In [ ]:
SCHEME_SHOW  = ["median-mad", "median-mad-noet"]     # any subset of SCHEMES
N_BATCH      = 12
MAX_VOX_PLOT = 300_000        # per subject, to keep the pooled histogram light

SCHEME_SHOW = [s for s in SCHEME_SHOW if s in CENTRE_SCALE or s in ("p99", "proposed", "oracle")]

rng_b = np.random.default_rng(SEED)
pick_b = rng_b.choice(len(usable), size=min(N_BATCH, len(usable)), replace=False)
loaded = []
for si in pick_b:
    try:
        loaded.append(load_subject(usable[int(si)]))
    except Exception as err:
        print(f"[skip] {type(err).__name__}")
n_ok = len(loaded)

fig, ax = plt.subplots(len(SCHEME_SHOW), 2, figsize=(12, 3.9 * len(SCHEME_SHOW)), squeeze=False)
for i, sch in enumerate(SCHEME_SHOW):
    xs_ = normalize(raw, brain, et, sch)[0]
    single = {c: xs_[..., c][brain] for c in range(4)}
    acc = {c: [] for c in range(4)}
    for r_, b_, e_ in loaded:
        x_ = normalize(r_, b_, e_, sch)[0]
        for c in range(4):
            v = x_[..., c][b_]
            acc[c].append(rng_b.choice(v, MAX_VOX_PLOT, replace=False)
                          if v.size > MAX_VOX_PLOT else v)
    batch = {c: np.concatenate(acc[c]) for c in range(4)}

    lo, hi = np.percentile(np.concatenate([batch[c] for c in range(4)]), [0.1, 99.9])
    bins = np.linspace(lo, hi, 160)
    for c, nm in enumerate(CONTRASTS):
        ax[i][0].hist(single[c], bins=bins, histtype="step", density=True, label=nm)
        ax[i][1].hist(batch[c], bins=bins, histtype="step", density=True, label=nm)
    ax[i][0].set_title(f"{sch}: one subject "
                       f"({os.path.basename(os.path.dirname(SUBJECT))})", fontsize=10)
    ax[i][1].set_title(f"{sch}: {n_ok} subjects pooled", fontsize=10)
    for a in ax[i]:
        a.axvline(0, color="k", lw=0.8)
        a.set_xlabel("normalized intensity"); a.grid(alpha=0.3); a.legend(fontsize=8)
    ax[i][0].set_ylabel("density")

    print(f"--- {sch}: {n_ok} subject(s) pooled, over brain voxels ---")
    lab_dl = f"p99/{DIV_SCALE:g}"
    print(f"{'contrast':<8} {'median':>9} {'p1':>9} {'p99':>9} {'min':>9} {'max':>9} "
          f"{lab_dl:>9}")
    for c, nm in enumerate(CONTRASTS):
        v = batch[c]
        print(f"{nm:<8} {np.median(v):>9.3f} {np.percentile(v, 1):>9.3f} "
              f"{np.percentile(v, 99):>9.3f} {v.min():>9.3f} {v.max():>9.3f} "
              f"{np.percentile(v, 99) / DIV_SCALE:>9.3f}")
    print()
plt.tight_layout(); plt.show()
print(f"The last column is what the dataloader hands the network after dividing by "
      f"scales={DIV_SCALE:g};\nfor an i2sb run in [-1, 1] that wants to sit near 1.")

## Caveats

* **The `-noet` schemes are fitted on the region two of the metrics are scored on.** `bias` and
  `bg_rms` are measured over `brain & ~ET`, which is exactly the sample the `-noet` centre and scale
  were estimated from. Rank on `et_med` and `sigma` instead -- and note that `separation` and `leak`
  are *not* a way around it, because both divide by a fit target: a scheme that flattens `bg_rms`
  inflates `separation` without the tumor contrast having moved at all.

* **If `median-mad-noet` matches `median-mad`, that is the answer, not a failed experiment.** MAD's
  50% breakdown point means a few percent of ET voxels should not move it. The estimator-shift
  section is there to make that verdict explicit rather than leaving it to be inferred from
  indistinguishable downstream tables.

* **You do not have an ET mask at inference.** BraTS ships segmentations for the training set, so
  the scheme is implementable for *training* -- but the T1ce divisor for a new subject is exactly
  what a synthesis model does not know, and here it would depend on a tumor mask derived from the
  T1ce you are trying to produce. This is the same problem the percentile family had. It only
  matters for the T1ce channel: `median-mad-noetce` leaves T1/FLAIR/T2 estimated over the full
  brain, so those stay computable at test time.

* **ET is one label.** `cmap.yaml` sets `et_labels: [4]`. Whole-tumor or edema would give a very
  different excluded region, and the necrotic core -- which is *dark* in T1ce -- is not in it, so
  the excluded set is not "everything abnormal", only "everything bright and abnormal".

* **Per-slice `-noet` is much noisier than per-volume `-noet`.** On a slice clipping the edge of the
  tumor a handful of ET voxels defines the excluded region, and `MIN_STAT_VOX` only guards the
  degenerate case where almost nothing is left. `PER_SLICE = False` (the default, matching
  `cmap.yaml`) pools the volume and avoids this entirely.

* **Slice-local scaling breaks comparability along $z$.** Each slice gets its own centre and scale,
  so the same tissue can take different values on neighbouring slices. Per-slice will always look
  flatter on the residual metrics -- more free parameters -- so judge the *switch* on downstream
  training, not on `bg_rms` here.

* **A smaller $\varsigma$ is not free.** A scheme that succeeds at flattening the healthy-tissue
  residual drives $\varsigma$ toward zero, and the faithful i2sb schedule cannot be centred below
  `std_fwd[-1] = 0.2416` because `linear_start = 1e-4` is hardcoded in `sb/base.py:i2sb_betas`. Check
  the `beta_max_hat` table before adopting a scheme, and fall back to the brownian schedule (free
  `tau`, no floor) if it lands under.